In [1]:
import sys
sys.path.append("..")
 
import json
from datasets import load_dataset
from src.utils import (
    score_tokens_perplexity,
    compress_random,
    compress_perplexity,
    build_training_sample
)

In [2]:
squad = load_dataset("rajpurkar/squad_v2", split="validation[:500]")

README.md:   0%|          | 0.00/8.92k [00:00<?, ?B/s]

squad_v2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 16.4MB            

squad_v2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

squad_v2/validation-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B / 1.35MB            

squad_v2/validation-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

In [3]:
import json, torch

def build_training_sample(context: str):
    result = score_tokens_perplexity(context)
    scores = torch.tensor(result["surprisal"])
    # min-max normalise to [0,1] → soft importance target
    targets = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
    return {"text": context, "tokens": result["tokens"], "targets": targets.tolist()}

samples = [build_training_sample(row["context"]) for row in squad]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [4]:
with open("../data/train_samples.json", "w") as f:
    json.dump(samples, f)

print(f"Saved {len(samples)} samples")

Saved 500 samples
